# Sprint 7 - Stronger backbones to push the field ceiling past 0.60 (v11/v12/v13)

**Why this sprint (see AGENT.md / learn.md):** MobileNetV2's best field-photo F1 is `both`'s
**0.5578** - every variant since has landed 0.41-0.56 on PlantDoc. To reach the app-grade target
**(field F1 >= 0.60, stretch 0.70)** we stop re-running the same backbone and try stronger
feature extractors. The PlantDoc paper itself reported ~0.70 accuracy with **ResNet-50** (on its
native classes/split; our mapped 38-class evaluation is harder, so treat 0.60 as the real gate).

The recipe is Sprint 4's fine-tune plus the fine-tune-then-mix warm start, but with a new
backbone, so no new training code:

1. **v11 `both_resnet50`** - two-stage field-strong recipe on ResNet-50 (Stage 1 PlantVillage
   head-only, then Stage 2 PlantDoc fine-tune + augmentation) - the ResNet-50 twin of `both`.
2. **v12 `both_efficientnet`** - same recipe on EfficientNet-B0.
3. **v13 `mixed_from_field_<winner>`** - the "fine-tune, then mix" move warm-started from the
   winning field-strong checkpoint, to re-learn the lab in one self-contained model.

Optional extra levers, used only if a variant lands just below a gate: `--tta` in evaluation
(test-time augmentation, ~1-3 points) and the label-audit script (Step 10) to find PlantDoc's
known mislabels, which cap every model.

**Why not two heads / a field toggle?** two-head needs to know a photo's domain to pick a head; a
toggle is a poor product and a style classifier is its own hard problem. A single mixed model has
no routing at all.


### Where we are (real runs, in the CSV)
- baseline (MobileNetV2): PlantVillage 0.9501 F1 | PlantDoc 0.1116 F1 (the gap)
- `both` (PD fine-tune only): PlantDoc **0.5578** F1 but forgot lab (0.3168)
- `mixed` (both together): PlantDoc 0.4107 F1 | PlantVillage **0.9592** F1
- `mixed_upsampled` (repeat 8): PlantDoc 0.4122 F1 | PlantVillage 0.9362 F1

**Sprint 7 gates (must BOTH pass on ONE checkpoint, no routing):**
1. **Field target:** PlantDoc F1 >= **0.60** (stretch 0.70) - what MobileNetV2 never reached.
2. **Lab recovered:** PlantVillage F1 >= 0.85 (the forgetting `both` suffered).


In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Requires the Sprint 0 archives on Drive (this is the first sprint whose checkpoint is NOT a
MobileNetV2, so Stage 1 must be re-trained per backbone - no warm start to reuse).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")    # durable archives (from Sprint 0)
LOCAL_RAW_DIR = Path("/content/folium_raw")             # per-session raw
LOCAL_DATA_DIR = Path("/content/folium_data")            # per-session organized splits
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

# Stage-2 tags use short names (stage2_efficientnet), so filenames cannot be built from
# the backbone variable (efficientnet_b0 -> _efficientnet_b0.pt). Map them explicitly.
STAGE2_CKPT = {
    "resnet50": "best_plantdoc_stage2_resnet50.pt",
    "efficientnet_b0": "best_plantdoc_stage2_efficientnet.pt",
}

def run(cmd, cwd, label):
    result = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{label}] failed (returncode {result.returncode})")
        print("stdout tail:\n", result.stdout[-2000:])
        print("stderr tail:\n", result.stderr[-2000:])
    assert result.returncode == 0, label
    return result

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as every sprint: unzip both archives, build `train/val/test` folders (seed 42, deterministic)
plus `class_map.json`. Idempotent.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("splits ready at", LOCAL_DATA_DIR)

## Step 4 - Stage 1 (PlantVillage, head-only) for both new backbones

Same as Sprint 1 but with `--backbone resnet50` / `--backbone efficientnet_b0`. Backbone frozen,
only the new head trains, no augmentation. These Stage 1 checkpoints are the warm start for the
field-strong fine-tunes in Steps 5-6.

Artifacts: `best_plantvillage_stage1_resnet50.pt` and `best_plantvillage_stage1_efficientnet.pt`.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--tag", "stage1_resnet50",
]
result = run(cmd, cwd=str(REPO_DIR), label="ml.train failed (stage1 resnet50)")
print("best Stage 1 (resnet50):", CHECKPOINT_DIR / "best_plantvillage_stage1_resnet50.pt")

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "efficientnet_b0",
    "--epochs", "5",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--tag", "stage1_efficientnet",
]
result = run(cmd, cwd=str(REPO_DIR), label="ml.train failed (stage1 efficientnet_b0)")
print("best Stage 1 (efficientnet_b0):", CHECKPOINT_DIR / "best_plantvillage_stage1_efficientnet.pt")

## Step 5 - v11: field-strong fine-tune on ResNet-50 (`both_resnet50`)

The Sprint 4 `both` recipe (Stage 2 PlantDoc fine-tune, last 2 blocks unfrozen, low backbone LR,
augmentation ON) but warm-started from the ResNet-50 Stage 1. If ResNet-50's extra capacity pays
off on real field photos, this is where the field F1 should climb past 0.56.

Artifacts: `best_plantdoc_stage2_resnet50.pt`. Header should show `backbone=resnet50`.

> If train accuracy shoots far past validation (overfitting 2,107 field images), re-run with
> `--unfreeze-blocks 1` (only the last block).

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--backbone", "resnet50",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1_resnet50.pt"),
    "--unfreeze-blocks", "2",
    "--lr", "1e-4",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--augment",
    "--tag", "stage2_resnet50",
]
result = run(cmd, cwd=str(REPO_DIR), label="ml.train failed (both_resnet50)")
print("best field-strong (resnet50):", CHECKPOINT_DIR / "best_plantdoc_stage2_resnet50.pt")

## Step 6 - v12: field-strong fine-tune on EfficientNet-B0 (`both_efficientnet`)

Same recipe, EfficientNet-B0 backbone. Artifacts: `best_plantdoc_stage2_efficientnet.pt`.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--backbone", "efficientnet_b0",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1_efficientnet.pt"),
    "--unfreeze-blocks", "2",
    "--lr", "1e-4",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--augment",
    "--tag", "stage2_efficientnet",
]
result = run(cmd, cwd=str(REPO_DIR), label="ml.train failed (both_efficientnet)")
print("best field-strong (efficientnet_b0):", CHECKPOINT_DIR / "best_plantdoc_stage2_efficientnet.pt")

## Step 7 - Evaluate v11/v12 on BOTH test sets (find the field winner)

Four rows: each backbone on `plantdoc_test` (the field ceiling) and `plantvillage_test` (did it
forget the lab like `both` did?). The backbone with the higher PlantDoc F1 wins Step 8.

> If a variant lands just below a gate, re-run with `--tta` appended to the evaluate command
> (test-time augmentation, ~1-3 points) before deciding.

In [ ]:
for backbone, variant in [("resnet50", "both_resnet50"), ("efficientnet_b0", "both_efficientnet")]:
    for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
        cmd = [
            sys.executable, "-m", "ml.evaluate",
            "--checkpoint", str(CHECKPOINT_DIR / STAGE2_CKPT[backbone]),
            "--data-dir", str(LOCAL_DATA_DIR),
            "--dataset", dataset,
            "--split", "test",
            "--results", str(RESULTS_DIR / "ablation_results.csv"),
            "--variant", variant,
        ] + extra
        result = run(cmd, cwd=str(REPO_DIR), label=f"ml.evaluate failed for {dataset}/{variant}")

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
field = df[(df["dataset"] == "plantdoc_test") & (df["variant"].isin(["both_resnet50", "both_efficientnet"]))]
print(field[["variant", "backbone", "accuracy", "precision", "recall", "f1"]].to_string(index=False))

if field.empty:
    print("\nNo field rows yet - run Step 7.")
else:
    winner = field.sort_values("f1", ascending=False).iloc[0]
    print(f"\nField winner: {winner['variant']} (PlantDoc F1 {winner['f1']:.4f})")
    if winner["f1"] >= 0.60:
        print("  >= 0.60 target: the backbone itself cleared the field gate.")
    else:
        print("  below 0.60: check v13 in Step 9 (and the label audit in Step 10).")

## Step 8 - v13: fine-tune-then-mix on the winning backbone (`mixed_from_field_<winner>`)

The fine-tune-then-mix move on the stronger backbone: warm-start from the field-strong winner, then train
mixed PlantVillage+PlantDoc head-only to re-learn the lab while keeping the field. Artifacts:
`best_plantvillage_mixed_from_field_<winner>.pt`.

Set `BACKBONE` to the winner printed in Step 7 (resnet50 or efficientnet_b0). `PLANTDOC_REPEAT`
sets the field share: 1 = PlantDoc's natural ~5% of each epoch, 8 = ~28% (the v13_x8 run that
tests whether a bigger field share stops the head collapsing back to the lab).

In [ ]:
BACKBONE = "resnet50"   # <- set to the Step 7 winner (resnet50 or efficientnet_b0)
PLANTDOC_REPEAT = 8   # field share: 1 = ~5% of mixed batches, 8 = ~28% (the v13_x8 run)
variant = f"mixed_from_field_{BACKBONE}" + (f"_x{PLANTDOC_REPEAT}" if PLANTDOC_REPEAT > 1 else "")

In [ ]:
x8_ckpt = CHECKPOINT_DIR / f"best_plantvillage_{variant}.pt"
if x8_ckpt.exists():
    resume_arg = ["--resume", str(x8_ckpt)]
else:
    resume_arg = ["--init-from", str(CHECKPOINT_DIR / STAGE2_CKPT[BACKBONE])]
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--mix-with", "plantdoc",
    "--plantdoc-repeat", str(PLANTDOC_REPEAT),
    "--backbone", BACKBONE,
    *resume_arg,
    "--lr", "1e-3",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--tag", variant,
]
result = run(cmd, cwd=str(REPO_DIR), label="ml.train failed (mixed_from_field)")
print("best single model:", x8_ckpt)

## Step 9 - Evaluate v13 on BOTH test sets (the Sprint 7 verdict)

Two rows: `plantdoc_test` (did the field survive lab re-learning?) and `plantvillage_test` (did
the lab recover from the field-strong start?). Add `--tta` if it is just below a gate.

In [ ]:
for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(CHECKPOINT_DIR / f"best_plantvillage_{variant}.pt"),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
        "--variant", variant,
    ] + extra
    result = run(cmd, cwd=str(REPO_DIR), label=f"ml.evaluate failed for {dataset}/{variant}")

## Step 10 - Label audit (optional but the highest-value next lever)

Runs the field-strong winner over the PlantDoc TRAIN split and writes every image with its
mapped class, predicted class, and confidence, sorted least-confident first. PlantDoc is
crowd-sourced and contains mislabeled / ambiguous photos; cleaning the worst of them is the
cheapest way past a field F1 plateau. Review `plantdoc_label_audit_<winner>.csv`, relabel or
drop the obvious errors, and re-run Step 8.

In [ ]:
result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "audit_plantdoc_labels.py"),
    "--checkpoint", str(CHECKPOINT_DIR / STAGE2_CKPT[BACKBONE]),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--split", "train",
    "--out", str(RESULTS_DIR / f"plantdoc_label_audit_{BACKBONE}.csv"),
], cwd=str(REPO_DIR), label="audit_plantdoc_labels.py failed")
print("audit CSV:", RESULTS_DIR / f"plantdoc_label_audit_{BACKBONE}.csv")

## Step 11 - The verdict: did a stronger backbone clear 0.60?

**Sprint 7 gates (must BOTH pass on ONE checkpoint, no routing):**
1. **Field target:** PlantDoc F1 >= **0.60** (stretch 0.70).
2. **Lab recovered:** PlantVillage F1 >= 0.85.

The full single-model landscape is what matters: how v11/v12/v13 sit relative to baseline,
`both`, `mixed`, and `mixed_upsampled` on BOTH datasets.

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
key = lambda v, d: df[(df["variant"] == v) & (df["dataset"] == d)]
print("The single-model landscape (PlantDoc F1 | PlantVillage F1):")
rows = [
    ("baseline            ", "baseline_pv_only_no_aug"),
    ("both (MobileNetV2)  ", "both"),
    ("both_resnet50       ", "both_resnet50"),
    ("both_efficientnet   ", "both_efficientnet"),
    ("mixed               ", "mixed"),
    ("mixed_from_field    ", "mixed_from_field"),
    ((f"v13_x{PLANTDOC_REPEAT}" if PLANTDOC_REPEAT > 1 else "v13").ljust(20), variant),
]
for label, v in rows:
    r_pd, r_pv = key(v, "plantdoc_test"), key(v, "plantvillage_test")
    a = f"{r_pd.iloc[0]['f1']:.4f}" if not r_pd.empty else "  -  "
    b = f"{r_pv.iloc[0]['f1']:.4f}" if not r_pv.empty else "  -  "
    print(f"  {label}  PlantDoc {a} | PlantVillage {b}")

pd_row, pv_row = key(variant, "plantdoc_test"), key(variant, "plantvillage_test")
if pd_row.empty or pv_row.empty:
    print(f"\n{variant}: missing rows - run Steps 8-9.")
else:
    pd_f1, pv_f1 = pd_row.iloc[0]["f1"], pv_row.iloc[0]["f1"]
    ok_field = pd_f1 >= 0.60
    ok_lab = pv_f1 >= 0.85
    print(f"\n  {variant}:  field >= 0.60 -> {'PASS' if ok_field else 'FAIL'} ({pd_f1:.4f}) | "
          f"lab >= 0.85 -> {'PASS' if ok_lab else 'FAIL'} ({pv_f1:.4f})")
    if ok_field and ok_lab:
        print("  Sprint 7 DONE: single model clears both gates, no routing.")
    elif ok_field:
        print("  Field target reached but lab recovered too weakly - see which classes the lab lost.")
    else:
        print("  Not there yet. Next: label audit (Step 10), more field photos, or the two-head router.")

## Step 12 - Predict field + lab photos (sanity check)

Classify a couple of field photos and lab photos with the best single model. Correct field labels
mean the field-strong warm start survived the lab re-learning.

In [ ]:
for folder in (LOCAL_DATA_DIR / "plantdoc" / "test", LOCAL_DATA_DIR / "plantvillage" / "test"):
    images = sorted(folder.glob("*/*.jpg"))[:2]
    for image in images:
        cmd = [
            sys.executable, "-m", "ml.predict",
            "--checkpoint", str(CHECKPOINT_DIR / f"best_plantvillage_{variant}.pt"),
            "--image", str(image),
            "--topk", "3",
        ]
        subprocess.run(cmd, cwd=str(REPO_DIR))
        print("  (true class folder:", image.parent.name, ")")

# Sprint 8 - Dual-head model (eliminate catastrophic forgetting)

**Why this sprint:** Sprint 7 proved the lab-vs-field trade-off is structural across 3
backbones. No single head can stay good at both domains because mixed training overwrites
the head on whichever domain dominates each epoch. The fix: two independent heads on the
same backbone, each trained on its own domain.

**Architecture:**
```
backbone (shared, frozen)
  ├── head_lab   → trained on PlantVillage only
  └── head_field → trained on PlantDoc only
```

**Inference:** run both heads, return the higher-confidence prediction per sample.

**Training flow:**
1. Stage 1: train head_lab on PlantVillage (backbone frozen)
2. Stage 2: freeze head_lab, train head_field on PlantDoc (backbone frozen)
3. Evaluate: both heads, confidence-based selection

## Step 13 - Train head_lab on PlantVillage (dual-head Stage 1)

Train only the lab head on PlantVillage. The backbone and field head stay frozen.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "lab",
    "--tag", "dual_head_lab",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="dual-head lab training failed")
print("Done. Lab head trained.")

## Step 14 - Train head_field on PlantDoc (dual-head Stage 2)

Load the dual-head checkpoint from Step 13. Freeze everything except head_field.
Train on PlantDoc mapped to the PlantVillage label space.

In [ ]:
DUAL_LAB_CKPT = CHECKPOINT_DIR / "best_plantvillage_dual_head_lab.pt"
assert DUAL_LAB_CKPT.exists(), f"Missing lab checkpoint: {DUAL_LAB_CKPT}. Run Step 13 first."

cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--backbone", "resnet50",
    "--epochs", "10",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "field",
    "--init-from", str(DUAL_LAB_CKPT),
    "--tag", "dual_head_field",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="dual-head field training failed")
print("Done. Field head trained. Both heads now in one checkpoint.")

## Step 15 - Evaluate dual-head on BOTH test sets

The dual-head model runs both heads and picks the higher-confidence prediction per sample.
This should give strong numbers on BOTH domains — no more trade-off.

In [ ]:
DUAL_FIELD_CKPT = CHECKPOINT_DIR / "best_plantvillage_dual_head_field.pt"
assert DUAL_FIELD_CKPT.exists(), f"Missing dual-head checkpoint: {DUAL_FIELD_CKPT}. Run Step 14 first."

for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(DUAL_FIELD_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--dual-head",
        "--variant", f"dual_head_resnet50_{dataset}",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ] + extra
    result = run(cmd, cwd=str(REPO_DIR), label=f"dual-head eval on {dataset} failed")
    print()

## Step 16 - The verdict: did dual-head break the trade-off?

**Sprint 8 gates (must BOTH pass):**
- PlantDoc F1 >= 0.60 (field gate)
- PlantVillage F1 >= 0.85 (lab gate)

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
keys = ["dual_head_resnet50_plantdoc", "dual_head_resnet50_plantvillage"]
dual_rows = df[df["variant"].isin(keys)].copy()
print(dual_rows[["variant", "f1"]].to_string(index=False))

if len(dual_rows) == 2:
    field_f1 = dual_rows[dual_rows["variant"] == "dual_head_resnet50_plantdoc"]["f1"].iloc[0]
    lab_f1 = dual_rows[dual_rows["variant"] == "dual_head_resnet50_plantvillage"]["f1"].iloc[0]
    field_pass = "PASS" if field_f1 >= 0.60 else "FAIL"
    lab_pass = "PASS" if lab_f1 >= 0.85 else "FAIL"
    print(f"\nDual-head verdict: field {field_f1:.4f} -> {field_pass} | lab {lab_f1:.4f} -> {lab_pass}")
else:
    print("\nDual-head rows not found. Run Step 15 first.")

## Where things live (Sprint 8)

**On Google Drive (durable):**
```
folium/checkpoints/best_plantvillage_dual_head_lab.pt     Stage 1: lab head trained
folium/checkpoints/best_plantvillage_dual_head_field.pt   Stage 2: field head trained
folium/results/ablation_results.csv                        All ablation rows
```

**Dual-head model:** one backbone + two independent heads. No catastrophic forgetting.
At inference, both heads run and the higher-confidence one wins per sample.